In [1]:
from __future__ import annotations
from enum import Enum
from typing import Dict, List, TypeVar, Set, Generator, Callable, Optional, Tuple

# --- Type Definitions ---
VertexID = TypeVar("VertexID")


class DFSEvent(Enum):
    """Defines the events that the DFS generator can yield."""

    ENTER = 1  # Pre-order: when a node is first discovered
    EXIT = 2  # Post-order: when a node's subtree has been fully explored


class CycleDetectedError(Exception):
    """Custom exception for cycle detection."""

    pass


def dfs_generator(
    start_node: VertexID,
    get_neighbors: Callable[[VertexID], List[VertexID]],
    *,
    visited: Optional[Set[VertexID]] = None,
    visiting: Optional[Set[VertexID]] = None,
    parent: Optional[VertexID] = None,
    depth: int = 0,
) -> Generator[Tuple[DFSEvent, VertexID, int, Optional[VertexID]], None, None]:
    """
    A powerful, reusable, generator-based DFS that yields events.

    This single function can be adapted for pre-order, post-order, path finding,
    cycle detection, and more by inspecting the yielded events.

    Args:
        start_node: The node to begin the traversal from.
        get_neighbors: A function that takes a node and returns its neighbors.
        visited: A set to track all fully explored nodes (for the entire graph).
        visiting: A set to track nodes on the current recursion path (for cycle detection).
        parent: The parent of the start_node in the DFS tree.
        depth: The depth of the start_node in the DFS tree.

    Yields:
        A tuple of (DFSEvent, node, depth, parent).
    """
    # Initialize state sets if this is the top-level call
    visited = visited if visited is not None else set()
    visiting = visiting if visiting is not None else set()

    # --- Pre-order Logic ---
    visited.add(start_node)
    visiting.add(start_node)
    yield (DFSEvent.ENTER, start_node, depth, parent)

    for neighbor in get_neighbors(start_node):
        if neighbor in visiting:
            # Back edge detected -> Cycle!
            # We can yield this event if the caller wants to handle it,
            # or raise an exception as is more common.
            raise CycleDetectedError(
                f"Cycle detected: edge from {start_node} to {neighbor}"
            )

        if neighbor not in visited:
            # Delegate to the recursive sub-generator
            yield from dfs_generator(
                neighbor,
                get_neighbors,
                visited=visited,
                visiting=visiting,
                parent=start_node,
                depth=depth + 1,
            )

    # --- Post-order Logic ---
    visiting.remove(start_node)
    yield (DFSEvent.EXIT, start_node, depth, parent)

In [2]:
def get_reachable_nodes(start_node: VertexID, graph: Dict) -> Set[VertexID]:
    get_neighbors = lambda node: graph.get(node, [])
    reachable = set()

    # We only care about the nodes we enter
    for event, node, _, _ in dfs_generator(start_node, get_neighbors):
        if event == DFSEvent.ENTER:
            reachable.add(node)

    return reachable

In [4]:
def find_path_dfs(
    start_node: VertexID, end_node: VertexID, graph: Dict
) -> Optional[List[VertexID]]:
    get_neighbors = lambda node: graph.get(node, [])
    path_stack = []

    try:
        for event, node, _, _ in dfs_generator(start_node, get_neighbors):
            if event == DFSEvent.ENTER:
                path_stack.append(node)
                if node == end_node:
                    return path_stack  # Found the path, return immediately
            elif event == DFSEvent.EXIT:
                path_stack.pop()
    except CycleDetectedError:  # Or handle as needed
        pass

    return None  # Path not found

In [ ]:
def topological_sort(graph: Dict) -> Optional[List[VertexID]]:
    get_neighbors = lambda node: graph.get(node, [])
    post_order = []
    visited = set()

    try:
        for node in list(graph.keys()):
            if node not in visited:
                for event, event_node, _, _ in dfs_generator(
                    node, get_neighbors, visited=visited
                ):
                    if event == DFSEvent.EXIT:
                        post_order.append(event_node)
    except CycleDetectedError:
        return None  # Topological sort not defined for cyclic graphs

    return post_order[::-1]

In [5]:
if __name__ == "__main__":
    # Example 1: A valid DAG for a topological sort
    # Represents course prerequisites
    dag_graph: Graph = {
        "Calculus II": ["Linear Algebra", "Differential Equations"],
        "Calculus I": ["Calculus II"],
        "Intro to CS": ["Data Structures", "Algorithms"],
        "Data Structures": ["Algorithms"],
        "Linear Algebra": ["Machine Learning"],
        "Algorithms": ["Machine Learning"],
        "Differential Equations": [],
        "Machine Learning": [],
    }

    # Add all nodes to the graph dictionary for completeness
    all_courses = {
        "Calculus II",
        "Linear Algebra",
        "Differential Equations",
        "Calculus I",
        "Intro to CS",
        "Data Structures",
        "Algorithms",
        "Machine Learning",
    }
    for course in all_courses:
        if course not in dag_graph:
            dag_graph[course] = []

    for course in all_courses:
        print(f"Reachable from {course}: {get_reachable_nodes(course, dag_graph)}")

        # for course in all_courses:
        path = find_path_dfs("Intro to CS", course, dag_graph)
        if path:
            print(f"Path from 'Intro to CS' to '{course}': {path}")
        else:
            print(f"No path found from 'Intro to CS' to '{course}'")

Reachable from Calculus II: {'Calculus II', 'Linear Algebra', 'Differential Equations', 'Machine Learning'}
No path found from 'Intro to CS' to 'Calculus II'
Reachable from Intro to CS: {'Data Structures', 'Intro to CS', 'Machine Learning', 'Algorithms'}
Path from 'Intro to CS' to 'Intro to CS': ['Intro to CS']
Reachable from Linear Algebra: {'Linear Algebra', 'Machine Learning'}
No path found from 'Intro to CS' to 'Linear Algebra'
Reachable from Algorithms: {'Machine Learning', 'Algorithms'}
Path from 'Intro to CS' to 'Algorithms': ['Intro to CS', 'Data Structures', 'Algorithms']
Reachable from Data Structures: {'Data Structures', 'Machine Learning', 'Algorithms'}
Path from 'Intro to CS' to 'Data Structures': ['Intro to CS', 'Data Structures']
Reachable from Differential Equations: {'Differential Equations'}
No path found from 'Intro to CS' to 'Differential Equations'
Reachable from Machine Learning: {'Machine Learning'}
Path from 'Intro to CS' to 'Machine Learning': ['Intro to CS', '